# Live interactive Beatles visuals

This page renders interactive charts directly in the website.

It targets the Beatles Spotify dataset from Kaggle and uses a public Beatles songwriter fallback in automated builds when the Kaggle CSV is not available locally.

In [ ]:
from pathlib import Path
import re
from typing import Optional

import pandas as pd
import plotly.express as px

KAGGLE_CSV = Path("projects/beatles-discography/data/beatles_spotify_dataset.csv")
PUBLIC_BASE_URL = "https://raw.githubusercontent.com/jarred13/The_Beatles_Recommendations/main/TheBeatlesCleaned.csv"
PUBLIC_SONGWRITER_URL = "https://raw.githubusercontent.com/inteligentni/Class-05-Feature-engineering/master/The%20Beatles%20songs%20dataset,%20v1,%20no%20NAs.csv"


def normalize_token(value: str) -> str:
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


def resolve_column(df: pd.DataFrame, candidates: list[str]) -> Optional[str]:
    mapping = {normalize_token(col): col for col in df.columns}
    for candidate in candidates:
        key = normalize_token(candidate)
        if key in mapping:
            return mapping[key]
    return None


def split_writers(value: str) -> list[str]:
    if pd.isna(value):
        return ["Unknown writer"]
    parts = re.split(r"\\s*(?:,|/|&| and )\\s*", str(value), flags=re.IGNORECASE)
    cleaned = [p.strip() for p in parts if p and p.strip()]
    return cleaned or ["Unknown writer"]


def load_dataset() -> tuple[pd.DataFrame, str]:
    if KAGGLE_CSV.exists():
        return pd.read_csv(KAGGLE_CSV), "local Kaggle CSV"
    return pd.read_csv(PUBLIC_BASE_URL), "public Beatles Spotify mirror"


In [ ]:
df, source_name = load_dataset()
song_col = resolve_column(df, ["song", "title", "track_name", "name", "Title"])
album_col = resolve_column(df, ["album", "album_name", "albumdebut", "Album.debut"])
writer_col = resolve_column(df, ["writers", "writer", "songwriter", "songwriters", "Songwriter"])
pop_col = resolve_column(df, ["popularity", "track_popularity", "spotify_popularity", "top_50_billboard", "top.50.billboard", "rank", "Top.50.Billboard"])

# Enrich writer/popularity fields when they are missing in the Spotify-only extract.
if writer_col is None or pop_col is None:
    ref = pd.read_csv(PUBLIC_SONGWRITER_URL)
    ref_song_col = resolve_column(ref, ["Title", "song", "title", "name"])
    ref_writer_col = resolve_column(ref, ["Songwriter", "writer", "writers"])
    ref_pop_col = resolve_column(ref, ["Top.50.Billboard", "top_50_billboard", "rank"])
    if song_col is None:
        raise ValueError("Could not identify song/title column in primary dataset.")

    df = df.copy()
    df["_join_song"] = df[song_col].astype(str).str.lower().str.strip()
    ref = ref.copy()
    ref["_join_song"] = ref[ref_song_col].astype(str).str.lower().str.strip()

    merged = ref[["_join_song", ref_writer_col, ref_pop_col]].drop_duplicates("_join_song")
    df = df.merge(merged, on="_join_song", how="left")
    df.drop(columns=["_join_song"], inplace=True)

    if writer_col is None:
        writer_col = ref_writer_col
    if pop_col is None:
        pop_col = ref_pop_col

if song_col is None or album_col is None or writer_col is None or pop_col is None:
    raise ValueError("Missing required columns for the interactive visuals.")

df = df.copy()
df[song_col] = df[song_col].fillna("Unknown song")
df[album_col] = df[album_col].fillna("Unknown album")

popularity_raw = pd.to_numeric(df[pop_col], errors="coerce")
if "billboard" in normalize_token(pop_col) or "rank" in normalize_token(pop_col):
    popularity_raw = popularity_raw.where(popularity_raw > 0)
    df["popularity_score"] = (51 - popularity_raw).clip(lower=0).fillna(0)
else:
    df["popularity_score"] = popularity_raw.fillna(popularity_raw.median()).fillna(0)

# Sunburst needs positive values for area sizing.
df["size_score"] = df["popularity_score"] + 1

writer_rows = df[[song_col, album_col, writer_col, "popularity_score", "size_score"]].copy()
writer_rows["writer_name"] = writer_rows[writer_col].apply(split_writers)
writer_rows = writer_rows.explode("writer_name")
writer_rows["writer_name"] = writer_rows["writer_name"].fillna("Unknown writer")

print(f"Data source used: {source_name}")
print(f"Rows: {len(df)} | Song column: {song_col} | Album column: {album_col} | Writer column: {writer_col} | Popularity column: {pop_col}")

In [ ]:
writer_summary = (
    writer_rows.groupby("writer_name", as_index=False)
    .agg(mean_popularity=("popularity_score", "mean"), song_count=(song_col, "nunique"))
    .sort_values("mean_popularity", ascending=False)
)

fig_writer = px.bar(
    writer_summary.head(20),
    x="writer_name",
    y="mean_popularity",
    color="song_count",
    title="Song popularity by writer / writers",
    labels={
        "writer_name": "Writer",
        "mean_popularity": "Average popularity",
        "song_count": "Songs",
    },
)
fig_writer.update_layout(template="plotly_white", xaxis_tickangle=-35)
fig_writer.show()

In [ ]:
fig_album_sunburst = px.sunburst(
    df,
    path=[album_col, song_col],
    values="size_score",
    color="popularity_score",
    color_continuous_scale="Viridis",
    title="Sunburst: full Beatles discography by album",
)
fig_album_sunburst.update_layout(template="plotly_white")
fig_album_sunburst.show()

In [ ]:
fig_writer_sunburst = px.sunburst(
    writer_rows,
    path=["writer_name", album_col, song_col],
    values="size_score",
    color="popularity_score",
    color_continuous_scale="Magma",
    title="Sunburst: full Beatles discography by writer",
)
fig_writer_sunburst.update_layout(template="plotly_white")
fig_writer_sunburst.show()